In [5]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

import shap
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("country_year_modeling_dataset_ready.csv") 
print(df.shape)
df.head()

(5712, 12)


,iso3c,country,year,mmr,gdp_pc,health_exp_gdp,fertility,skilled_births,pm25,heat_index35_days,cooling_degree_days,log_mmr
0,AFE,Africa Eastern and Southern,2000,612.0,706.727261,5.657523,5.549893,40.977528,26.783513,0.0,3061.85,6.418365
1,AFE,Africa Eastern and Southern,2001,586.0,625.827815,5.803267,5.501766,98.500000,26.809926,0.0,3255.66,6.375025
2,AFE,Africa Eastern and Southern,2002,564.0,630.512328,5.408213,5.449696,98.900000,26.718091,0.0,3474.73,6.336826
3,AFE,Africa Eastern and Southern,2003,537.0,815.432296,5.989670,5.407328,98.900000,26.580603,0.0,3506.74,6.287859
4,AFE,Africa Eastern and Southern,2004,516.0,989.015464,6.069927,5.381308,98.500000,26.475208,0.0,3467.88,6.248043


In [9]:
features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days"
]

features = [f for f in features if f in df.columns]
print("Features:", features)

X = df[features].copy()
y = df["log_mmr"].copy() if "log_mmr" in df.columns else np.log1p(df["mmr"])

Features: ['gdp_pc', 'health_exp_gdp', 'fertility', 'skilled_births', 'pm25', 'heat_index35_days', 'cooling_degree_days']


In [11]:
split_year = 2018

train = df[df["year"] <= split_year].copy()
test  = df[df["year"] > split_year].copy()

X_train = train[features]
y_train = train["log_mmr"] if "log_mmr" in train.columns else np.log1p(train["mmr"])

X_test = test[features]
y_test = test["log_mmr"] if "log_mmr" in test.columns else np.log1p(test["mmr"])

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (4522, 7) Test: (1190, 7)


In [13]:
best_model = GradientBoostingRegressor(random_state=42)
best_model.fit(X_train, y_train)

preds_log = best_model.predict(X_test)


preds_mmr = np.expm1(preds_log)
true_mmr  = np.expm1(y_test)

print("MAE (MMR scale):", mean_absolute_error(true_mmr, preds_mmr))
print("RMSE (MMR scale):", np.sqrt(mean_squared_error(true_mmr, preds_mmr)))
print("R2 (log scale):", r2_score(y_test, preds_log))

MAE (MMR scale): 64.26546341359467
RMSE (MMR scale): 120.12362978407579
R2 (log scale): 0.7412383518029251


In [ ]:
# SHAP (beeswarm + bar)
X_bg = X_train.sample(min(500, len(X_train)), random_state=42)
X_explain = X_test.sample(min(1000, len(X_test)), random_state=42)

explainer = shap.Explainer(best_model, X_bg)
shap_values = explainer(X_explain)

shap.plots.beeswarm(shap_values, max_display=12)
shap.plots.bar(shap_values, max_display=12)